# 320. Custom Dataset을 이용한 Hugging Face BERT model Fine Tuning (Pure PyTorch)

- NAVER Movie review dataset을 이용하여 transformers BERT model을 fine tuning  → 감성분석 모델 작성

- **순수 PyTorch 학습 루프**를 이용한 Fine Tuning (Trainer API 미사용)

- Colab gpu 사용

In [1]:
# !pip install transformers[torch]

In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from torch.utils.data import DataLoader
from torch.optim import AdamW
from transformers import get_linear_schedule_with_warmup
import torch.nn.functional as F
import torch
import pandas as pd
from tqdm.auto import tqdm

In [3]:
import urllib.request
import os

data_dir = os.path.join(os.path.expanduser("~"), ".keras", "datasets")
os.makedirs(data_dir, exist_ok=True)

DATA_TRAIN_PATH = os.path.join(data_dir, "ratings_train.txt")
DATA_TEST_PATH = os.path.join(data_dir, "ratings_test.txt")

if not os.path.exists(DATA_TRAIN_PATH):
    urllib.request.urlretrieve(
        "https://raw.github.com/ironmanciti/Infran_NLP/master/data/naver_movie/ratings_train.txt",
        DATA_TRAIN_PATH)
    print("ratings_train.txt 다운로드 완료")

if not os.path.exists(DATA_TEST_PATH):
    urllib.request.urlretrieve(
        "https://raw.github.com/ironmanciti/Infran_NLP/master/data/naver_movie/ratings_test.txt",
        DATA_TEST_PATH)
    print("ratings_test.txt 다운로드 완료")

print("데이터 준비 완료:", DATA_TRAIN_PATH)

ratings_train.txt 다운로드 완료
ratings_test.txt 다운로드 완료
데이터 준비 완료: /root/.keras/datasets/ratings_train.txt


### Train Set

In [4]:
#  학습 데이터 로드
train_data = pd.read_csv(DATA_TRAIN_PATH, delimiter='\t')

print(train_data.shape)
train_data.head()

(150000, 3)


,id,document,label
0,9976970,아 더빙.. 진짜 짜증나네요 목소리,0
1,3819312,흠...포스터보고 초딩영화줄....오버연기조차 가볍지 않구나,1
2,10265843,너무재밓었다그래서보는것을추천한다,0
3,9045019,교도소 이야기구먼 ..솔직히 재미는 없다..평점 조정,0
4,6483659,사이몬페그의 익살스런 연기가 돋보였던 영화!스파이더맨에서 늙어보이기만 했던 커스틴 ...,1


In [5]:
# 결측값(NaN)이 포함된 행을 모두 제거
train_data.dropna(inplace=True)

# 현재 DataFrame의 구조 요약 출력
train_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 149995 entries, 0 to 149999
Data columns (total 3 columns):
 #   Column    Non-Null Count   Dtype 
---  ------    --------------   ----- 
 0   id        149995 non-null  int64 
 1   document  149995 non-null  object
 2   label     149995 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 4.6+ MB


### Test Set

In [6]:
#  검증 데이터 로드
test_data = pd.read_csv(DATA_TEST_PATH, delimiter='\t')

print(test_data.shape)
test_data.head()

(50000, 3)


,id,document,label
0,6270596,굳 ㅋ,1
1,9274899,GDNTOPCLASSINTHECLUB,0
2,8544678,뭐야 이 평점들은.... 나쁘진 않지만 10점 짜리는 더더욱 아니잖아,0
3,6825595,지루하지는 않은데 완전 막장임... 돈주고 보기에는....,0
4,6723715,3D만 아니었어도 별 다섯 개 줬을텐데.. 왜 3D로 나와서 제 심기를 불편하게 하죠??,0


In [7]:
# 결측값(NaN)이 포함된 행을 모두 제거
test_data.dropna(inplace=True)
test_data.info()

<class 'pandas.core.frame.DataFrame'>
Index: 49997 entries, 0 to 49999
Data columns (total 3 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   id        49997 non-null  int64 
 1   document  49997 non-null  object
 2   label     49997 non-null  int64 
dtypes: int64(2), object(1)
memory usage: 1.5+ MB


- 훈련 시간 단축을 위해 1/10 의 data 만 sampling - 6분 소요

In [8]:
# 훈련 데이터에서 무작위로 15,000개 샘플 추출 (재현성을 위해 random_state 고정)
df_train = train_data.sample(n=15000, random_state=1)

# 테스트 데이터에서 무작위로 5,000개 샘플 추출
df_test = test_data.sample(n=5000, random_state=1)

# 추출된 데이터프레임의 행과 열 크기 출력
print(df_train.shape)  # (15000, 열 수)
print(df_test.shape)   # (5000, 열 수)

(15000, 3)
(5000, 3)


In [9]:
# 훈련 데이터의 'label' 열에 있는 각 클래스(레이블)별 개수를 집계
df_train['label'].value_counts()

,count
label,
0,7524
1,7476


In [10]:
# 훈련 데이터에서 입력 문장(document)과 레이블(label)을 리스트로 추출
X_train = df_train['document'].values.tolist()      # 입력 텍스트 (리스트 형태)
y_train = df_train['label'].values.tolist()               # 정답 레이블 (리스트 형태)

# 테스트 데이터에서도 동일하게 입력과 레이블을 리스트로 추출
X_test = df_test['document'].values.tolist()    # 입력 텍스트 (리스트 형태)
y_test = df_test['label'].values.tolist()             # 정답 레이블 (리스트 형태)

## pre-trained bert model 호출
### tokenizer 호출
- 토큰화 처리를 합니다. bert 다국어 version 용의 pre-trained tokenizer 를 불러옵니다.

In [11]:
# 사전학습된 KLUE BERT 토크나이저 불러오기
# 'klue/bert-base'는 한국어 자연어 처리를 위해 특화된 한국어 BERT 모델로,
# KLUE(Korean Language Understanding Evaluation) 벤치마크에서 우수한 성능을 나타냄
tokenizer = AutoTokenizer.from_pretrained('klue/bert-base')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/425 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

pre-trained tokenizer 를 이용하여 train set 과 test set 을 token 화 합니다.

- Input IDs : 토큰 인덱스, 모델에서 입력으로 사용할 시퀀스를 구축하는 토큰의 숫자 표현
- Token Type IDs : 한 쌍의 문장 또는 질문 답변에 대한 분류 시 사용  
- attention mask : `1`은 주목해야 하는 값을 나타내고 `0`은 패딩된 값을 나타냅니다.  
```
[CLS] SEQUENCE_A [SEP] SEQUENCE_B [SEP]
ex) [CLS] HuggingFace is based in NYC [SEP] Where is HuggingFace based? [SEP]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 1, 1, 1]
```

In [12]:
# 훈련 데이터(X_train)를 BERT 입력 형식에 맞게 토크나이즈
# - truncation=True: 최대 길이를 초과하는 문장은 자동으로 자름
# - padding=True: 짧은 문장은 최대 길이에 맞춰 0으로 패딩
train_encodings = tokenizer(X_train, truncation=True, padding=True)

# 테스트 데이터(X_test)도 동일한 방식으로 토크나이즈
test_encodings = tokenizer(X_test, truncation=True, padding=True)

In [13]:
# 토크나이징된 훈련 데이터의 키 목록 확인
# 일반적으로 'input_ids', 'attention_mask', (선택적으로 'token_type_ids')가 포함됨
print(type(train_encodings))

<class 'transformers.tokenization_utils_base.BatchEncoding'>


In [14]:
print(train_encodings['input_ids'][0])
print(train_encodings['attention_mask'][0])
print(train_encodings['token_type_ids'][0])

[2, 12, 20609, 2446, 2373, 2048, 21, 13, 7929, 2179, 2147, 4281, 2116, 1556, 1500, 2066, 35, 25687, 10814, 2292, 19521, 16, 5690, 7285, 1088, 2114, 12074, 1202, 2088, 3804, 8969, 10993, 3656, 13014, 2073, 25735, 2097, 2182, 18, 612, 3125, 2470, 16, 7094, 2470, 16, 11607, 2570, 31221, 5690, 7285, 3732, 3853, 2088, 1088, 2114, 12074, 1462, 2170, 3656, 5690, 2366, 2170, 2259, 4073, 31221, 11051, 12, 4386, 13, 3885, 6858, 2125, 18, 18, 3, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0]
[0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0

### Convert encodings to Tensors

- 레이블과 인코딩을 Dataset 개체로 변환합니다. Pytorch를 이용합니다.  

- PyTorch에서 이것은 `torch.utils.data.Dataset` 객체를 하고 `__len__` 및 `__getitem__`을 구현하여 수행됩니다.


In [15]:
# PyTorch Dataset 클래스를 상속하여 IMDb 감성 분석용 커스텀 데이터셋 정의
class IMDbDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels=None):
        # 토크나이즈된 입력 (input_ids, attention_mask 등) 저장
        self.encodings = encodings
        # 정답 레이블 (선택사항)
        self.labels = labels

    def __getitem__(self, idx):
        # 주어진 인덱스(idx)에 해당하는 데이터 추출
        # encodings 딕셔너리에서 각 항목별로 같은 인덱스를 추출하고 텐서로 변환
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        # 레이블이 있는 경우 함께 반환
        if self.labels:
            item["labels"] = torch.tensor(self.labels[idx])
        return item

    def __len__(self):
        # 데이터셋의 전체 샘플 수 반환
        return len(self.encodings["input_ids"])

# 훈련용 PyTorch Dataset 객체 생성
train_dataset = IMDbDataset(train_encodings, y_train)

# 테스트용 PyTorch Dataset 객체 생성
test_dataset = IMDbDataset(test_encodings, y_test)

### PyTorch 학습 설정

- Training warmup steps :  

    - 이는 일반적으로 설정된 수의 훈련 단계(워밍업 단계)에 대해 매우 낮은 학습률을 사용한다는 것을 의미합니다. 워밍업 단계 후에 "일반" 학습률 또는 학습률 스케줄러를 사용합니다. 또한 워밍업 단계 수에 따라 학습률을 점진적으로 높일 수 있습니다.

- weight_decay : 가중치 감쇠. L2 regularization

In [16]:
# 하이퍼파라미터 설정
NUM_EPOCHS = 2
TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 16
WARMUP_STEPS = 500
WEIGHT_DECAY = 0.01
LEARNING_RATE = 5e-5
LOGGING_STEPS = 100

# DataLoader 생성
train_loader = DataLoader(train_dataset, batch_size=TRAIN_BATCH_SIZE, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=EVAL_BATCH_SIZE, shuffle=False)

### model Train

Colab 에서 약 17분 소요

In [17]:
import time

# 디바이스 설정
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"사용 디바이스: {device}")

# 사전학습된 KLUE BERT 모델 로드
model = AutoModelForSequenceClassification.from_pretrained(
    'klue/bert-base',
    num_labels=2
)
model.to(device)

# weight_decay를 bias와 LayerNorm에는 적용하지 않음
no_decay = ['bias', 'LayerNorm.weight']
optimizer_grouped_parameters = [
    {'params': [p for n, p in model.named_parameters() if not any(nd in n for nd in no_decay)],
     'weight_decay': WEIGHT_DECAY},
    {'params': [p for n, p in model.named_parameters() if any(nd in n for nd in no_decay)],
     'weight_decay': 0.0}
]

optimizer = AdamW(optimizer_grouped_parameters, lr=LEARNING_RATE)

total_steps = len(train_loader) * NUM_EPOCHS
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=WARMUP_STEPS, num_training_steps=total_steps)

# 학습 시작
s = time.time()
global_step = 0
running_loss = 0.0

model.train()
for epoch in range(NUM_EPOCHS):
    progress_bar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{NUM_EPOCHS}")
    for batch in progress_bar:
        batch = {k: v.to(device) for k, v in batch.items()}

        outputs = model(**batch)
        loss = outputs.loss

        loss.backward()
        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()

        running_loss += loss.item()
        global_step += 1

        if global_step % LOGGING_STEPS == 0:
            avg_loss = running_loss / LOGGING_STEPS
            progress_bar.set_postfix({'loss': f'{avg_loss:.4f}', 'step': global_step})
            running_loss = 0.0

print(f"\n학습 완료! 총 {global_step} steps")

사용 디바이스: cuda


model.safetensors:   0%|          | 0.00/445M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: klue/bert-base
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.embeddings.position_ids               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on you

Epoch 1/2:   0%|          | 0/1875 [00:00<?, ?it/s]

Epoch 2/2:   0%|          | 0/1875 [00:00<?, ?it/s]


학습 완료! 총 3750 steps


In [18]:
print("경과 시간 : {:.2f}분".format((time.time() - s)/60))

경과 시간 : 11.89분


In [19]:
# 테스트 데이터셋을 사용하여 모델 성능 평가
model.eval()
total_eval_loss = 0.0
num_batches = 0

with torch.no_grad():
    for batch in tqdm(test_loader, desc="평가 중"):
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        total_eval_loss += outputs.loss.item()
        num_batches += 1

avg_eval_loss = total_eval_loss / num_batches
print(f"평가 손실 (eval_loss): {avg_eval_loss:.4f}")

평가 중:   0%|          | 0/313 [00:00<?, ?it/s]

평가 손실 (eval_loss): 0.3207


In [20]:
# 테스트 데이터셋에 대해 예측 수행
import numpy as np

model.eval()
all_logits = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(test_loader, desc="예측 중"):
        labels = batch.pop("labels")
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        all_logits.append(outputs.logits.cpu())
        all_labels.append(labels)

all_logits = torch.cat(all_logits, dim=0)
all_labels = torch.cat(all_labels, dim=0)

print(f"예측 완료: logits shape = {all_logits.shape}, labels shape = {all_labels.shape}")

예측 중:   0%|          | 0/313 [00:00<?, ?it/s]

예측 완료: logits shape = torch.Size([5000, 2]), labels shape = torch.Size([5000])


fine-tuned model 은 logit 을 return

In [21]:
# 모델의 분류기(classifier) 층 확인
model.classifier

Linear(in_features=768, out_features=2, bias=True)

In [22]:
# 예측 결과에서 로짓(logits) 확인
y_logit = all_logits
y_logit[:10]

tensor([[ 3.2782, -3.1101],
        [ 0.6916, -0.4259],
        [ 0.1502, -0.7682],
        [ 3.1856, -2.7296],
        [-1.9464,  1.6142],
        [ 3.3038, -3.2805],
        [ 0.5232, -0.9818],
        [ 3.3377, -3.4011],
        [-3.1445,  1.9286],
        [ 0.7759, -0.7153]])

In [23]:
# 소프트맥스 함수를 사용해 각 샘플의 클래스별 확률을 계산
y_pred = F.softmax(y_logit, dim=-1).argmax(axis=1).numpy()

# 예측된 레이블 중 앞 30개를 리스트로 출력
print(list(y_pred[:30]))

# 실제 정답 레이블 중 앞 30개를 출력
print(y_test[:30])

[np.int64(0), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(1), np.int64(0), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(1), np.int64(0), np.int64(1)]
[0, 0, 1, 0, 1, 0, 1, 0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 1]


In [24]:
from sklearn.metrics import confusion_matrix, accuracy_score

# 예측값과 실제 정답 사이의 정확도(accuracy)를 계산
print(accuracy_score(y_test, y_pred))

# 혼동 행렬(confusion matrix) 계산
# 실제 레이블과 예측 레이블을 비교하여 각 클래스별 예측 결과를 표로 요약
cm = confusion_matrix(y_test, y_pred)
cm

0.8814


array([[2190,  305],
       [ 288, 2217]])

In [27]:
# 예측할 문장
x = "돈주고 보기에는 아까운 영화 ㅠㅠ..."
# x = "돈이 안 아깝지 않은 멋진 영화 ㅎㅎ..."

# 1. 입력 토크나이즈
inputs = tokenizer([x], truncation=True, padding=True, return_tensors="pt")

# 2. 입력을 디바이스로 이동하고 예측
model.eval()
inputs = {k: v.to(device) for k, v in inputs.items()}

with torch.no_grad():
    outputs = model(**inputs)
    logits = outputs.logits

# 3. 소프트맥스 -> 확률 -> argmax
probs = F.softmax(logits, dim=-1)
pred = torch.argmax(probs, dim=1).item()

# 4. 결과 출력
print("긍정" if pred == 1 else "부정")

긍정


# Next Step
20 만개 전체 dataset으로 fine tuning